# Prompt engineering con Gemma y gestión de contexto en Google Colab

Este cuaderno presenta una introducción teórica y práctica al *prompt engineering* aplicada a modelos de lenguaje de gran tamaño, con especial atención al uso de Gemma mediante la API de Google y al mantenimiento de contexto mediante una estructura sencilla inspirada en el enfoque de LangGraph.

El objetivo no es únicamente mostrar cómo invocar un modelo, sino enseñar cómo diseñar instrucciones de calidad, cómo estructurar interacciones de varios turnos y cómo transformar una petición imprecisa en una especificación operativa. Para ello, cada bloque combina explicación conceptual, código ejecutable y un ejemplo inmediatamente posterior.

El cuaderno está diseñado para ejecutarse en **Google Colab** y asume que el estudiante introducirá manualmente su clave de API en una de las primeras celdas como variable global.


## 1. Qué es un LLM

Un **Large Language Model** (LLM) es un modelo estadístico entrenado sobre grandes volúmenes de texto con el propósito de predecir la continuación más plausible de una secuencia lingüística. Aunque en la práctica se perciba como un sistema que “responde preguntas”, “resume”, “clasifica” o “redacta”, en sentido técnico su operación fundamental consiste en estimar qué token o fragmento lingüístico resulta más probable dado un contexto previo.

Los LLM modernos se apoyan en arquitecturas de tipo **Transformer**, cuya principal ventaja es la capacidad de modelar dependencias contextuales a gran escala. Gracias a mecanismos como la atención, el modelo no trata cada palabra de forma aislada, sino que pondera la relevancia de distintos fragmentos del contexto para producir una salida coherente. De este modo, una instrucción no es simplemente un texto de entrada, sino una configuración explícita del problema que el modelo debe resolver.

Desde una perspectiva didáctica, conviene entender que un LLM no “comprende” en sentido humano ni razona como una persona. Lo que hace es operar sobre patrones aprendidos. Precisamente por ello, la calidad de la instrucción de entrada tiene un impacto decisivo: si el problema está mal formulado, el modelo tenderá a producir una respuesta gramaticalmente sólida pero conceptualmente pobre, ambigua o poco útil.


### Ejemplo

Si se pide a un LLM “háblame de liderazgo”, el modelo debe inferir por sí mismo el nivel técnico, la audiencia, el propósito y el formato deseado. En cambio, si se le indica “explica tres principios de liderazgo para mandos intermedios, en tono profesional y con un ejemplo aplicado a gestión de equipos”, el espacio de ambigüedad se reduce y la respuesta tiende a ser más pertinente.


## 2. Qué es el prompt engineering

El *prompt engineering* puede definirse como la disciplina de diseñar instrucciones eficaces para orientar el comportamiento de un modelo de lenguaje hacia un resultado útil, controlado y verificable. En sus primeras fases se entendía a menudo como un conjunto de trucos o recetas. Sin embargo, en un enfoque más maduro debe considerarse una práctica de especificación: consiste en convertir una necesidad difusa en una tarea bien definida.

Un buen prompt no se mide por su longitud ni por su aparente sofisticación retórica. Se mide por su capacidad para precisar el encargo. En la práctica, esto suele implicar identificar con claridad al menos cinco elementos: la **tarea**, el **contexto**, las **restricciones**, el **formato de salida** y el **criterio de calidad**. Cuanto más explícitos sean estos elementos, menor será la carga inferencial que se traslada al modelo.

Esta idea es especialmente importante en entornos profesionales. En usos casuales, una respuesta aproximada puede resultar aceptable. En cambio, en documentación, análisis, automatización, soporte, formación o toma de decisiones, la vaguedad se traduce en error, trabajo adicional o resultados inconsistentes.


### Ejemplo

Una instrucción como “ayúdame con una estrategia” es insuficiente porque deja sin definir el dominio, la audiencia, el horizonte temporal, los criterios de éxito y la forma de entrega. El *prompt engineering* consiste precisamente en descomponer esa petición general en una especificación utilizable.


## 3. Preparación del entorno en Google Colab

Antes de trabajar con prompts, es necesario preparar el entorno de ejecución. En este cuaderno se utilizará el SDK oficial de Google para interactuar con modelos generativos y, adicionalmente, una librería orientada a la gestión de estado y memoria para construir interacciones de varios turnos.

Desde el punto de vista pedagógico, conviene separar dos capas. La primera es la capa de **inferencia**, es decir, la llamada al modelo. La segunda es la capa de **orquestación**, que incluye historial, memoria, preferencias del usuario y reglas de interacción. Una enseñanza importante en aplicaciones reales es que llamar a un modelo no basta: también hay que diseñar el flujo conversacional que rodea a esa llamada.


In [ ]:
!pip -q install -U google-genai langgraph langchain-core ipywidgets


### Ejemplo

Este bloque instala las dependencias mínimas para ejecutar el cuaderno. La biblioteca `google-genai` se utilizará para las llamadas al modelo, mientras que `langgraph` servirá como referencia práctica para estructurar contexto y memoria dentro de un flujo conversacional.


## 4. Configuración de la clave de API como variable global

En este cuaderno la clave de API se introducirá manualmente en una de las primeras celdas. Esta decisión simplifica la ejecución didáctica, ya que permite al estudiante comprender de forma explícita qué variable gobierna la autenticación con el servicio.

A efectos de diseño del cuaderno, conviene centralizar la clave y el nombre del modelo en variables globales. Esto facilita la mantenibilidad: si más adelante se desea cambiar el modelo o reutilizar la misma configuración en distintas funciones, no será necesario modificar múltiples celdas.


In [ ]:
GEMINI_API_KEY = 'PEGA_AQUI_TU_API_KEY'
MODEL_NAME = 'gemma-4-31b-it'

assert GEMINI_API_KEY and GEMINI_API_KEY != 'PEGA_AQUI_TU_API_KEY', 'Debes introducir una API key válida'
print('Configuración global cargada')


### Ejemplo

El estudiante únicamente debe sustituir el texto provisional por su clave válida. A partir de ese momento, el resto del cuaderno reutilizará esta variable global, lo que evita repetir credenciales en varios puntos del código.


## 5. Creación del cliente y primer contacto con el modelo

Una vez definida la clave, el siguiente paso consiste en crear el cliente de acceso al modelo. Conceptualmente, esta es la interfaz entre el cuaderno y la infraestructura de inferencia. A partir de aquí, cada llamada al modelo será una operación parametrizada por un contenido de entrada y, opcionalmente, por configuraciones adicionales.

En términos didácticos, este primer contacto debe ser deliberadamente simple. Antes de diseñar prompts complejos conviene observar qué ocurre cuando se lanza una instrucción básica. Esa comparación inicial será útil más adelante, cuando se introduzcan mejoras estructurales en el diseño del prompt.


In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents='Explica brevemente qué es el prompt engineering.'
)

print(response.text)


### Ejemplo

Este bloque realiza una llamada mínima al modelo. El interés pedagógico de esta primera prueba no reside en la calidad final de la respuesta, sino en establecer un punto de referencia: se trata de observar cómo responde el modelo cuando la instrucción es funcional, pero aún no está cuidadosamente diseñada.


## 6. Estructura conceptual de un buen prompt

Un prompt eficaz suele organizarse alrededor de una estructura relativamente estable. Una formulación útil para enseñanza es la siguiente:

1. **Tarea**: qué debe hacer el modelo.
2. **Contexto**: para quién, en qué dominio y con qué propósito.
3. **Restricciones**: tono, longitud, límites, exclusiones o cautelas.
4. **Formato**: cómo debe presentarse la respuesta.
5. **Criterio de calidad**: qué condiciones debe cumplir la salida para ser considerada satisfactoria.

Esta estructura no es una plantilla rígida, pero sí un marco práctico. Su valor radica en que desplaza el esfuerzo desde la improvisación hacia la explicitación. Muchas respuestas mediocres no se deben a que el modelo sea incapaz, sino a que el encargo está subdefinido.


### Ejemplo

Si se desea un resumen para dirección, no basta con pedir “resume esto”. Resulta preferible indicar: finalidad ejecutiva, extensión limitada, tono neutro y foco en implicaciones prácticas. Cada una de estas piezas reduce un tipo distinto de ambigüedad.


## 7. Comparación entre un prompt genérico y un prompt bien especificado

La forma más clara de comprender la diferencia entre un prompt débil y uno robusto es compararlos empíricamente. Un prompt genérico obliga al modelo a inferir demasiado; un prompt estructurado delimita el problema.

Esta comparación es importante porque permite al estudiante desarrollar criterio. El objetivo no es memorizar una fórmula, sino reconocer qué tipo de información falta cuando una petición resulta imprecisa.


In [ ]:
prompt_malo = 'Háblame de prompt engineering.'

prompt_bueno = '''
Tarea: explica qué es el prompt engineering.
Contexto: la audiencia son profesionales sin formación técnica profunda.
Restricciones: utiliza un tono formal y didáctico; evita jerga innecesaria; no superes 220 palabras.
Formato: responde con una introducción breve, cuatro ideas clave y un ejemplo final.
Criterio de calidad: la explicación debe ser clara, reutilizable en contexto profesional y fácil de estudiar.
'''

for etiqueta, prompt in [('PROMPT_MALO', prompt_malo), ('PROMPT_BUENO', prompt_bueno)]:
    print(f'===== {etiqueta} =====')
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    print(response.text)
    print('' + '=' * 100 + '')


### Ejemplo

Al comparar las salidas, suele observarse que el segundo prompt produce una respuesta más estable y pedagógica. La razón no es misteriosa: el modelo recibe una especificación explícita sobre audiencia, tono, estructura y calidad esperada.


## 8. Encapsular el diseño del prompt en una función reutilizable

Cuando una estructura de prompt demuestra ser útil, conviene convertirla en una función. Esta decisión tiene valor tanto técnico como metodológico. Técnicamente, reduce repetición y facilita mantenimiento. Metodológicamente, obliga a pensar el prompting como un proceso de diseño parametrizable.

En términos pedagógicos, encapsular un prompt en una función ayuda a distinguir entre **contenido variable** y **andamiaje estable**. La tarea concreta cambiará; la lógica de especificación puede mantenerse.


In [ ]:
def ask_gemma(task, context='', constraints='', output_format='', quality=''):
    prompt = f'''
Tarea: {task}
Contexto: {context}
Restricciones: {constraints}
Formato: {output_format}
Criterio de calidad: {quality}
'''
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text


### Ejemplo


In [ ]:
respuesta = ask_gemma(
    task='Resume los principios del prompt engineering contemporáneo',
    context='La audiencia son analistas de negocio en fase inicial de adopción de IA',
    constraints='Usa tono formal, no excedas seis viñetas y evita tecnicismos innecesarios',
    output_format='Lista de viñetas con una frase explicativa por punto',
    quality='Cada punto debe incluir una recomendación práctica aplicable'
)

print(respuesta)


## 9. Few-shot prompting: enseñar mediante ejemplos

El *few-shot prompting* consiste en proporcionar al modelo varios ejemplos de entrada y salida antes de pedirle una nueva resolución. Su utilidad radica en que algunos criterios son difíciles de formular como reglas abstractas, pero sencillos de mostrar mediante casos ilustrativos.

Esta técnica es especialmente valiosa en tareas de clasificación, etiquetado, reformulación, extracción o normalización. Desde un punto de vista pedagógico, enseña una idea central: cuando el comportamiento esperado tiene matices, mostrar ejemplos concretos suele ser más eficaz que multiplicar instrucciones ambiguas.


In [ ]:
few_shot_prompt = '''
Clasifica mensajes de clientes en una de estas categorías: urgente, normal, baja prioridad.

Ejemplo 1:
Mensaje: 'El sistema de pagos está caído y no podemos vender.'
Etiqueta: urgente

Ejemplo 2:
Mensaje: '¿Podéis reenviarme la factura del mes pasado?'
Etiqueta: normal

Ejemplo 3:
Mensaje: 'Cuando tengáis tiempo, me gustaría conocer nuevas funcionalidades.'
Etiqueta: baja prioridad

Ahora clasifica el siguiente mensaje y responde solo con la etiqueta:
'Múltiples usuarios reportan que no pueden iniciar sesión desde esta mañana.'
'''

print(client.models.generate_content(model=MODEL_NAME, contents=few_shot_prompt).text)


### Ejemplo

En este caso, los ejemplos no solo ilustran el formato, sino también el criterio de decisión. El modelo aprende por analogía operativa cuál es la frontera entre urgencia, normalidad y baja prioridad.


## 10. Salidas estructuradas y control del resultado

Una de las prácticas más importantes en aplicaciones reales es exigir salidas estructuradas. Un texto libre puede ser correcto desde el punto de vista lingüístico y, sin embargo, resultar poco útil para un flujo automatizado o para una revisión consistente. En cambio, un resultado estructurado reduce ambigüedad, facilita validación y permite reutilización programática.

Desde una perspectiva metodológica, esta práctica desplaza parte del control desde la interpretación posterior hacia la propia formulación de la tarea. El prompt no solo define qué pensar, sino también cómo entregar el resultado.


In [ ]:
structured_prompt = '''
Analiza la iniciativa: implantar un asistente interno para soporte de recursos humanos.

Devuelve exclusivamente un JSON válido con esta estructura:
{
  "beneficios": ["..."],
  "riesgos": ["..."],
  "recomendacion": "..."
}
'''

print(client.models.generate_content(model=MODEL_NAME, contents=structured_prompt).text)


### Ejemplo

Este patrón es útil cuando la salida debe integrarse en aplicaciones, hojas de cálculo, cuadros de mando o procesos de evaluación posteriores. La estructura exigida impone una disciplina sobre el resultado.


## 11. Chain-of-Thought: razonamiento paso a paso

Una de las técnicas más relevantes en prompt engineering para tareas complejas es el **Chain-of-Thought**. Su principio es sencillo: en lugar de pedir al modelo únicamente una respuesta final, se le solicita que descomponga el problema en pasos intermedios de razonamiento antes de formular la conclusión. Esta técnica resulta especialmente útil cuando la tarea exige análisis multietapa, comparación de criterios, inferencia causal o resolución de problemas no triviales.

Desde un punto de vista didáctico, el valor del Chain-of-Thought no reside únicamente en mejorar el resultado final, sino también en hacer visible el proceso lógico que conduce a ese resultado. Esto permite al estudiante inspeccionar la respuesta, detectar errores intermedios y comprender mejor por qué una salida puede ser correcta o incorrecta. En otras palabras, convierte un resultado opaco en un proceso parcialmente auditable.

No obstante, esta técnica debe utilizarse con criterio. No se trata de obligar al modelo a producir razonamientos largos e innecesarios para cualquier tarea, sino de emplearla cuando el problema lo justifique. Para tareas simples, una instrucción breve suele bastar. Para tareas complejas, en cambio, pedir una descomposición explícita puede reducir omisiones y mejorar la consistencia.


In [ ]:
direct_prompt = '''
Analiza qué estrategia debería seguir una pequeña empresa SaaS que sufre alta rotación de clientes.
Responde en un único párrafo.
'''

cot_prompt = '''
Analiza qué estrategia debería seguir una pequeña empresa SaaS que sufre alta rotación de clientes.

Trabaja paso a paso.
1. Identifica posibles causas de la rotación.
2. Agrupa esas causas en factores de producto, precio, onboarding y soporte.
3. Evalúa cuáles parecen más plausibles en una empresa SaaS pequeña.
4. Propón una estrategia priorizada.
5. Cierra con una recomendación final breve y ejecutiva.
'''

for etiqueta, prompt in [('RESPUESTA_DIRECTA', direct_prompt), ('CHAIN_OF_THOUGHT', cot_prompt)]:
    print(f'===== {etiqueta} =====')
    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    print(response.text)
    print('' + '=' * 100 + '')


### Ejemplo

Al comparar ambos resultados, suele observarse que la versión con razonamiento paso a paso no solo ofrece una recomendación, sino también una estructura analítica más transparente. Esto hace que la respuesta sea más útil para tareas de evaluación y aprendizaje.


## 12. Explorar varias soluciones y seleccionar la más adecuada

Otra familia de técnicas relevantes consiste en no conformarse con una única solución inicial, sino pedir al modelo que genere varias alternativas, las evalúe y seleccione la más adecuada con criterios explícitos. Este enfoque está relacionado con estrategias como *self-consistency* y con esquemas más deliberativos como *Tree of Thoughts*, en los que el modelo explora distintos caminos de razonamiento antes de converger en una respuesta final.

Pedagógicamente, esta técnica es importante porque introduce una distinción fundamental entre **producir opciones** y **evaluar opciones**. En muchos contextos profesionales, no basta con que el modelo proponga una idea aparentemente razonable; es necesario comparar alternativas, ponderar ventajas y riesgos, y justificar por qué una de ellas resulta preferible.

Esta práctica puede entenderse como una forma básica de deliberación estructurada. El modelo primero amplía el espacio de soluciones posibles y después aplica un marco de evaluación. Así se evita, al menos parcialmente, el sesgo hacia la primera respuesta plausible que aparezca.


In [ ]:
multi_solution_prompt = '''
Debes proponer una estrategia para mejorar la adopción de una herramienta interna de IA en una empresa mediana.

Sigue este proceso:
1. Genera 3 soluciones distintas.
2. Para cada solución, evalúa: coste, complejidad de implantación, impacto esperado y riesgo.
3. Resume la evaluación en una tabla.
4. Selecciona la solución más adecuada.
5. Explica brevemente por qué supera a las demás.
'''

response = client.models.generate_content(model=MODEL_NAME, contents=multi_solution_prompt)
print(response.text)


### Ejemplo

En este patrón, el interés no se limita a la solución elegida. También importa la evaluación comparativa previa, porque permite comprender qué criterios han motivado la selección final y en qué condiciones podría ser preferible otra alternativa.


## 13. Introducción al contexto y a la memoria conversacional

Hasta este punto, cada interacción se ha tratado como una petición relativamente aislada. Sin embargo, muchos usos reales de un LLM se desarrollan en varios turnos. En ese escenario aparece una cuestión central: cómo conservar el contexto necesario sin obligar al usuario a repetir continuamente la misma información.

La gestión de contexto puede entenderse en dos niveles. El primero es el **historial reciente**, es decir, los últimos turnos de la conversación. El segundo es una **memoria más estable**, donde pueden almacenarse preferencias, reglas, objetivos o datos persistentes del usuario. Distinguir ambos niveles es esencial para diseñar sistemas más útiles y menos repetitivos.

En este cuaderno se utilizará una implementación sencilla con almacenamiento en memoria para fines docentes. El objetivo no es agotar la complejidad del diseño conversacional, sino mostrar cómo integrar contexto de forma comprensible y reutilizable.


In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(index={"embed": lambda texts: [[float(len(t)), 1.0] for t in texts], "dims": 2})
namespace = ('student', 'prompt_course')

store.put(namespace, 'preferences', {
    'tone': 'formal y académico',
    'audience': 'estudiantes que se inician en el uso profesional de LLMs',
    'preferred_format': 'explicaciones breves con ejemplos posteriores'
})

prefs = store.get(namespace, 'preferences')
prefs.value


### Ejemplo

Aquí se registran preferencias relativamente estables. En una aplicación real, este tipo de memoria puede incluir nivel del usuario, idioma preferente, objetivos del curso, restricciones de formato o cualquier otra información que convenga conservar entre turnos.


## 12. Construcción de un sistema básico de chat con contexto acumulado

El siguiente paso consiste en combinar historial reciente y memoria estable dentro de una misma función de interacción. Conceptualmente, esto equivale a construir una capa mínima de orquestación. El modelo no responderá solo a la pregunta actual, sino a la pregunta actual reinterpretada a la luz del contexto disponible.

Desde una perspectiva formativa, esta sección es importante porque muestra que un sistema de prompting serio rara vez se reduce a una sola cadena de texto. Más bien se parece a un ensamblaje de componentes: preferencias, historial, reglas, entradas del usuario y llamada final al modelo.


In [ ]:
conversation = []

def chat_with_context(user_message):
    prefs = store.get(namespace, 'preferences').value
    history_text = ''.join([f"{m['role']}: {m['content']}" for m in conversation[-6:]])

    prompt = f'''
Eres un asistente docente especializado en prompt engineering.

Debes responder con estas preferencias:
- Tono: {prefs['tone']}
- Audiencia: {prefs['audience']}
- Formato preferido: {prefs['preferred_format']}

Historial reciente:
{history_text}

Nueva petición del usuario:
{user_message}

Instrucción: ofrece una respuesta coherente con el historial y con las preferencias indicadas.
'''

    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    answer = response.text
    conversation.append({'role': 'user', 'content': user_message})
    conversation.append({'role': 'assistant', 'content': answer})
    return answer


### Ejemplo


In [ ]:
print(chat_with_context('Explícame qué distingue un prompt aceptable de un prompt excelente.'))
print('' + '=' * 80 + '')
print(chat_with_context('Ahora adapta esa explicación a un caso de marketing B2B.'))


## 13. Pedir aclaraciones antes de formular un prompt final

Una de las prácticas más valiosas en prompt engineering consiste en no cerrar prematuramente el problema cuando la petición inicial todavía está subdefinida. En muchos casos, la calidad del resultado no depende tanto de la potencia del modelo como de la calidad del proceso previo de especificación. Si la solicitud es ambigua, el sistema debería comportarse de manera metodológica: primero detectar vacíos de información, después solicitar aclaraciones y solo finalmente construir un prompt final.

Desde el punto de vista didáctico, esta etapa tiene un interés especial porque enseña a transformar una necesidad difusa en una instrucción operativa. En lugar de fijar artificialmente un número de preguntas y respuestas, resulta más útil permitir que el sistema genere un conjunto variable de aclaraciones según la complejidad del caso. Del mismo modo, conviene que las respuestas del alumno se proporcionen en un único bloque de texto, con el formato natural de preguntas y respuestas, porque ese patrón se aproxima mejor a un flujo real de trabajo.

En esta sección se implementará un sistema sencillo de refinamiento en dos fases. En la primera, el modelo analizará la petición inicial y formulará un número arbitrario de preguntas aclaratorias, apoyándose además en el contexto disponible. En la segunda, el estudiante introducirá un bloque de texto que contenga esas preguntas y sus respectivas respuestas. A partir de ese bloque, el sistema generará un prompt final más preciso y justificará por qué la nueva formulación es superior a la petición original.


In [ ]:
refinement_state = {
    'initial_request': '',
    'clarification_questions': '',
    'qa_block': ''
}


def request_clarifications(initial_request):
    refinement_state['initial_request'] = initial_request
    refinement_state['qa_block'] = ''

    prefs = store.get(namespace, 'preferences').value
    history_text = ''.join([f"{m['role']}: {m['content']}" for m in conversation[-6:]])

    prompt = f'''
Eres un asistente docente especializado en prompt engineering.

Preferencias del entorno:
- Tono: {prefs['tone']}
- Audiencia: {prefs['audience']}
- Formato preferido: {prefs['preferred_format']}

Historial reciente:
{history_text}

Petición inicial del estudiante:
{initial_request}

Tu tarea es identificar qué información falta para poder redactar un prompt de alta calidad.
Formula únicamente las preguntas aclaratorias necesarias. El número de preguntas debe depender de la ambigüedad real del caso.
Escríbelas en una lista numerada y no redactes todavía el prompt final.
'''

    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    questions_text = response.text.strip()
    refinement_state['clarification_questions'] = questions_text
    return questions_text



def build_prompt_from_qa_block(qa_block):
    refinement_state['qa_block'] = qa_block

    store.put(namespace, 'latest_refinement', {
        'initial_request': refinement_state['initial_request'],
        'clarification_questions': refinement_state['clarification_questions'],
        'qa_block': refinement_state['qa_block']
    })

    prefs = store.get(namespace, 'preferences').value
    refinement_memory = store.get(namespace, 'latest_refinement').value
    history_text = ''.join([f"{m['role']}: {m['content']}" for m in conversation[-6:]])

    prompt = f'''
Eres un especialista en diseño de prompts.

Preferencias:
- Tono: {prefs['tone']}
- Audiencia: {prefs['audience']}
- Formato preferido: {prefs['preferred_format']}

Historial reciente:
{history_text}

Petición inicial del estudiante:
{refinement_memory['initial_request']}

Preguntas aclaratorias formuladas previamente:
{refinement_memory['clarification_questions']}

Bloque de preguntas y respuestas aportado por el estudiante:
{refinement_memory['qa_block']}

Redacta ahora un prompt final de alta calidad.
Debe incluir de forma explícita: tarea, contexto, restricciones, formato y criterio de calidad.
Después del prompt, añade una breve explicación académica de por qué esta versión mejora la petición inicial.
'''

    response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
    return response.text


### Ejemplo: fase 1, solicitud de aclaraciones

El siguiente bloque genera las preguntas que el sistema considera necesarias. El número no está prefijado: depende de cuánta información falte realmente en la petición inicial.


In [ ]:
print(request_clarifications('Necesito un prompt para analizar clientes.'))


### Ejemplo: fase 2, construcción del prompt final a partir de un bloque de preguntas y respuestas

Tras revisar las preguntas generadas, el alumno puede responder en un único bloque de texto. No es necesario pasar cada respuesta como argumento independiente. Basta con mantener un formato legible, por ejemplo:

Pregunta 1: ...  
Respuesta 1: ...  
Pregunta 2: ...  
Respuesta 2: ...

El siguiente bloque muestra un ejemplo completo.


In [ ]:
qa_block = '''
Pregunta 1: ¿Cuál es el objetivo del análisis de clientes?
Respuesta 1: Segmentar la base para priorizar acciones de retención.

Pregunta 2: ¿Qué datos están disponibles?
Respuesta 2: Frecuencia de compra, gasto medio, recencia, canal y antigüedad.

Pregunta 3: ¿Qué formato de salida necesitas?
Respuesta 3: Una tabla con segmentos, criterios de clasificación y acciones sugeridas.

Pregunta 4: ¿Para qué tipo de lector debe prepararse el resultado?
Respuesta 4: Para un equipo de marketing con perfil de negocio.
'''

final_prompt = build_prompt_from_qa_block(qa_block)
print(final_prompt)


## 14. Agrupación final en una clase reutilizable

Una vez aislados los componentes fundamentales, conviene integrarlos en una clase compacta que reúna configuración, memoria, historial, generación de aclaraciones y construcción del prompt final. Esta integración es metodológicamente importante porque refleja mejor cómo se diseñan sistemas reales: no como funciones dispersas, sino como componentes coherentes que comparten estado y reglas.

En esta versión final, la clase no solo podrá solicitar aclaraciones, sino también generar el prompt refinado a partir de un bloque de preguntas y respuestas. De este modo, el alumno dispone de una abstracción completa para reproducir el flujo entero de refinamiento.


In [ ]:
class GemmaPromptAssistant:
    def __init__(self, api_key, model='gemma-4-31b-it'):
        self.client = genai.Client(api_key=api_key)
        self.model = model
        self.store = InMemoryStore(index={"embed": lambda texts: [[float(len(t)), 1.0] for t in texts], "dims": 2})
        self.namespace = ('user', 'session')
        self.history = []
        self.refinement_state = {
            'initial_request': '',
            'clarification_questions': '',
            'qa_block': ''
        }

    def set_preferences(self, tone, audience, preferred_format):
        self.store.put(self.namespace, 'preferences', {
            'tone': tone,
            'audience': audience,
            'preferred_format': preferred_format
        })

    def _get_prefs(self):
        prefs = self.store.get(self.namespace, 'preferences')
        return prefs.value if prefs else {
            'tone': 'formal',
            'audience': 'general',
            'preferred_format': 'explicación breve'
        }

    def _history_text(self):
        return ''.join([f"{m['role']}: {m['content']}" for m in self.history[-6:]])

    def ask(self, task):
        prefs = self._get_prefs()
        prompt = f'''
Tarea: {task}
Tono: {prefs['tone']}
Audiencia: {prefs['audience']}
Formato preferido: {prefs['preferred_format']}
Historial reciente:
{self._history_text()}
'''
        response = self.client.models.generate_content(model=self.model, contents=prompt)
        answer = response.text
        self.history.append({'role': 'user', 'content': task})
        self.history.append({'role': 'assistant', 'content': answer})
        return answer

    def request_clarifications(self, initial_request):
        prefs = self._get_prefs()
        self.refinement_state['initial_request'] = initial_request
        self.refinement_state['qa_block'] = ''

        prompt = f'''
Eres un asistente docente especializado en diseño de prompts.

Tono: {prefs['tone']}
Audiencia: {prefs['audience']}
Formato preferido: {prefs['preferred_format']}
Historial reciente:
{self._history_text()}

Petición inicial:
{initial_request}

Formula únicamente las preguntas aclaratorias necesarias para mejorar la petición.
El número de preguntas debe depender de la ambigüedad del caso.
Preséntalas en lista numerada y no redactes aún el prompt final.
'''
        response = self.client.models.generate_content(model=self.model, contents=prompt)
        questions = response.text
        self.refinement_state['clarification_questions'] = questions
        return questions

    def build_prompt_from_qa_block(self, qa_block):
        prefs = self._get_prefs()
        self.refinement_state['qa_block'] = qa_block

        prompt = f'''
Eres un especialista en prompt engineering.

Tono: {prefs['tone']}
Audiencia: {prefs['audience']}
Formato preferido: {prefs['preferred_format']}
Historial reciente:
{self._history_text()}

Petición inicial:
{self.refinement_state['initial_request']}

Preguntas aclaratorias formuladas:
{self.refinement_state['clarification_questions']}

Bloque de preguntas y respuestas del usuario:
{qa_block}

Redacta un prompt final de alta calidad.
Debe incluir: tarea, contexto, restricciones, formato y criterio de calidad.
Después, añade una explicación breve de por qué este prompt es mejor que la petición original.
'''
        response = self.client.models.generate_content(model=self.model, contents=prompt)
        answer = response.text
        self.history.append({'role': 'user', 'content': self.refinement_state['initial_request']})
        self.history.append({'role': 'assistant', 'content': answer})
        return answer


### Ejemplo


In [ ]:
assistant = GemmaPromptAssistant(GEMINI_API_KEY, model=MODEL_NAME)
assistant.set_preferences(
    tone='formal y académico',
    audience='profesionales que comienzan a trabajar con LLMs',
    preferred_format='explicación estructurada con ejemplo final'
)

print(assistant.ask('Explica cómo diseñar un prompt para resumir informes extensos.'))
print('' + '-' * 80 + '')

questions = assistant.request_clarifications('Necesito un prompt para analizar reseñas de clientes.')
print(questions)
print('' + '-' * 80 + '')

qa_block = '''
Pregunta 1: ¿Cuál es el objetivo del análisis?
Respuesta 1: Detectar temas recurrentes y oportunidades de mejora.

Pregunta 2: ¿Qué tipo de reseñas se analizarán?
Respuesta 2: Reseñas textuales de clientes de un comercio electrónico.

Pregunta 3: ¿Qué formato de salida deseas?
Respuesta 3: Una tabla con temas, frecuencia, sentimiento y recomendación de acción.
'''

print(assistant.build_prompt_from_qa_block(qa_block))


## 15. Síntesis final

A lo largo del cuaderno se ha mostrado que el prompt engineering no debe entenderse como una colección de fórmulas superficiales, sino como una práctica de especificación orientada a reducir ambigüedad y aumentar control. También se ha visto que, en contextos reales, una llamada al modelo rara vez basta por sí sola: suele ser necesario combinarla con historial, memoria, preguntas aclaratorias y estructuras de salida bien definidas.

Desde el punto de vista formativo, la idea central puede resumirse así: un buen sistema basado en LLMs no se limita a “preguntar cosas a un modelo”, sino que diseña cuidadosamente las condiciones en las que ese modelo recibe, interpreta y transforma una petición. El paso de una solicitud vaga a un prompt final bien construido constituye, en sí mismo, una parte esencial del trabajo intelectual.


## 16. Ejercicio propuesto: diseño avanzado de un sistema de prompting

Se propone un ejercicio de nivel avanzado cuyo objetivo no es únicamente ejecutar llamadas al modelo, sino **diseñar, justificar, comparar y evaluar** distintas estrategias de prompting sobre un mismo problema aplicado.

### Enunciado

Imagina que trabajas como responsable de IA aplicada en una organización educativa que quiere utilizar Gemma para asistir en tres tareas distintas relacionadas con estudiantes universitarios:

1. **Clasificación** de consultas recibidas por correo en categorías operativas.
2. **Generación** de respuestas breves y útiles para el estudiante.
3. **Análisis estructurado** de riesgos o limitaciones antes de automatizar completamente el proceso.

Tu trabajo consiste en construir una pequeña solución experimental basada en prompts, analizar su comportamiento y justificar las decisiones tomadas.

### Datos de partida

Utiliza al menos estos seis mensajes de ejemplo como conjunto mínimo de prueba:

- “No puedo acceder al campus virtual desde anoche y mañana tengo una entrega.”
- “Quiero saber si puedo reconocer créditos por actividades deportivas.”
- “Me equivoqué al subir el archivo de la práctica y el plazo terminó hace una hora.”
- “¿Dónde puedo consultar las fechas de solicitud de beca para el próximo curso?”
- “Llevo dos semanas esperando respuesta sobre mi cambio de grupo.”
- “No entiendo cómo se calcula la nota final de esta asignatura.”

Puedes añadir más ejemplos si lo consideras conveniente.

### Tareas obligatorias

**1. Diseño de prompts**

Diseña **tres prompts distintos** para resolver la clasificación de mensajes. Deben corresponder, como mínimo, a estos enfoques:

- un prompt **simple o baseline**;
- un prompt con **few-shot prompting**;
- un prompt con **restricciones explícitas de formato y criterio de decisión**.

En todos los casos, define tú mismo las categorías de clasificación. Deben ser razonables, mutuamente distinguibles y útiles para una organización educativa.

**2. Evaluación comparativa**

Ejecuta los tres prompts sobre el mismo conjunto de mensajes y compara los resultados.

Debes analizar, al menos:

- consistencia de etiquetas;
- claridad de la salida;
- facilidad de reutilización en una aplicación real;
- errores, ambigüedades o clasificaciones discutibles.

No basta con decir cuál “parece mejor”: debes justificarlo con observaciones concretas.

**3. Generación de respuesta al estudiante**

Escoge **dos mensajes** del conjunto y diseña un prompt que genere una respuesta institucional breve, útil y empática.

La respuesta debe cumplir simultáneamente estas condiciones:

- tono formal pero cercano;
- máximo 120 palabras;
- no inventar normativas que no se hayan proporcionado;
- distinguir claramente entre información segura y recomendación prudente.

Después, explica por qué tu prompt reduce el riesgo de alucinación o respuestas excesivamente genéricas.

**4. Salida estructurada**

Diseña un prompt que devuelva, para cada mensaje, un **JSON válido** con esta estructura mínima:

```json
{
  "mensaje": "...",
  "categoria": "...",
  "prioridad": "alta/media/baja",
  "requiere_revision_humana": true,
  "justificacion": "..."
}
```

Ejecuta el prompt y comprueba si la salida se ajusta realmente al formato pedido. Si falla, documenta el fallo y modifica el prompt para mejorarlo.





In [ ]:
# ==============================
# EJERCICIO AVANZADO PROPUESTO
# ==============================

mensajes_estudiantes = [
    "No puedo acceder al campus virtual desde anoche y mañana tengo una entrega.",
    "Quiero saber si puedo reconocer créditos por actividades deportivas.",
    "Me equivoqué al subir el archivo de la práctica y el plazo terminó hace una hora.",
    "¿Dónde puedo consultar las fechas de solicitud de beca para el próximo curso?",
    "Llevo dos semanas esperando respuesta sobre mi cambio de grupo.",
    "No entiendo cómo se calcula la nota final de esta asignatura."
]

print("Mensajes de partida cargados correctamente:")
for i, m in enumerate(mensajes_estudiantes, 1):
    print(f"{i}. {m}")

# A partir de aquí, el alumno debe:
# 1) diseñar tres prompts distintos de clasificación,
# 2) compararlos,
# 3) generar respuestas institucionales,
# 4) exigir salida JSON válida,
# 5) redactar una reflexión crítica final.

